In [1]:
import pandas as pd

# If refering to Excel, use: pd.read_excel('[name].xlsx', sheet_name='XXX')
df_fossil_inflows = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/Fossil inflows.csv')

df_fossil_inflows.head()



,Flow,Added info,Amount in dataset,Unit in dataset,Amount_tonnes,Carrier,Type,Source
0,"Crude oil, NGL, refinery feedstocks, additives...",Indigenous production,19057.03,thousand_tonnes,19057031.0,Crude oil,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
1,"Crude oil, NGL, refinery feedstocks, additives...",Imports,497079.54,thousand_tonnes,497079537.0,Crude oil,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
2,"Crude oil, NGL, refinery feedstocks, additives...",Exports,14406.38,thousand_tonnes,14406378.0,Crude oil,Outflow,https://ec.europa.eu/eurostat/databrowser/view...
3,"Crude oil, NGL, refinery feedstocks, additives...",Change in stock,4042.54,thousand_tonnes,4042544.0,Crude oil,Stock,https://ec.europa.eu/eurostat/databrowser/view...
4,Oil products,Imports,288398.08,thousand_tonnes,288398077.0,Refined oil,Inflow,https://ec.europa.eu/eurostat/databrowser/view...


In [2]:
df_fossil_inflows_clean = df_fossil_inflows.drop(columns=['Amount in dataset', 'Unit in dataset', 'Source'])

df_fossil_inflows_clean.tail()
df_fossil_inflows_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Flow           64 non-null     str    
 1   Added info     64 non-null     str    
 2   Amount_tonnes  64 non-null     float64
 3   Carrier        64 non-null     str    
 4   Type           64 non-null     str    
dtypes: float64(1), str(4)
memory usage: 2.6 KB


In [3]:

df_ccf = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/CCF.csv')
ccf_clean = df_ccf[['Carrier', 'Carbon content factor [kg C/kg]']].copy()
ccf_clean['Carbon content factor [kg C/kg]'] = pd.to_numeric(
    ccf_clean['Carbon content factor [kg C/kg]'], 
    errors='coerce'
)
ccf_clean = ccf_clean.dropna(subset=['Carbon content factor [kg C/kg]'])
ccf_clean

,Carrier,Carbon content factor [kg C/kg]
1,Hard coal,0.716
2,Lignite,0.328
3,Natural gas,0.734
4,Crude oil,0.846
5,Refined oil,0.856
6,Petroleum coke,0.863
7,Coke (from coal),0.823
8,Petrochemical feedstock,0.869
10,Primary biomass,0.500
11,Wood pellets / Timber of Biomass forestry,0.476


In [4]:
# Map and Merge so "Gas" flows find "Natural gas" factors
mapping = {
    'Gas': 'Natural gas', 
    'Crude oil': 'Crude oil', 
    'Hard coal': 'Hard coal', 
    'Lignite': 'Lignite', 
    'Refined oil': 'Refined oil'
}
df_fossil_inflows_clean['CCF_Match'] = df_fossil_inflows_clean['Carrier'].map(mapping)

# Merge the 'Carbon content factor' column to fossil data
df_final = df_fossil_inflows_clean.merge(ccf_clean, left_on='CCF_Match', right_on='Carrier', how='left', suffixes=('', '_ccf'))

df_final['Amount_tonnes'] = pd.to_numeric(df_final['Amount_tonnes'], errors='coerce')
df_final['Carbon content factor [kg C/kg]'] = pd.to_numeric(df_final['Carbon content factor [kg C/kg]'], errors='coerce')

# 2. Now perform the multiplication
df_final['tonnes_Carbon'] = df_final['Amount_tonnes'] * df_final['Carbon content factor [kg C/kg]']


# Save the result
#df_final.to_csv('Calculated_Fossil_Carbon_Fixed.csv', index=False)
df_final.info()
df_final.head()





<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Flow                             64 non-null     str    
 1   Added info                       64 non-null     str    
 2   Amount_tonnes                    64 non-null     float64
 3   Carrier                          64 non-null     str    
 4   Type                             64 non-null     str    
 5   CCF_Match                        64 non-null     str    
 6   Carrier_ccf                      64 non-null     str    
 7   Carbon content factor [kg C/kg]  64 non-null     float64
 8   tonnes_Carbon                    64 non-null     float64
dtypes: float64(3), str(6)
memory usage: 4.6 KB


,Flow,Added info,Amount_tonnes,Carrier,Type,CCF_Match,Carrier_ccf,Carbon content factor [kg C/kg],tonnes_Carbon
0,"Crude oil, NGL, refinery feedstocks, additives...",Indigenous production,19057031.0,Crude oil,Inflow,Crude oil,Crude oil,0.846,1.612225e+07
1,"Crude oil, NGL, refinery feedstocks, additives...",Imports,497079537.0,Crude oil,Inflow,Crude oil,Crude oil,0.846,4.205293e+08
2,"Crude oil, NGL, refinery feedstocks, additives...",Exports,14406378.0,Crude oil,Outflow,Crude oil,Crude oil,0.846,1.218780e+07
3,"Crude oil, NGL, refinery feedstocks, additives...",Change in stock,4042544.0,Crude oil,Stock,Crude oil,Crude oil,0.846,3.419992e+06
4,Oil products,Imports,288398077.0,Refined oil,Inflow,Refined oil,Refined oil,0.856,2.468688e+08


In [5]:
df_final_fossil_inflow = df_final.drop(columns=['Amount_tonnes', 'CCF_Match', 'Carrier_ccf', 'Carbon content factor [kg C/kg]'])
df_final_fossil_inflow.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Calculated_Fossil_Carbon_Inflow.csv', index=False)